# 🏈 NFL Player Prop Feature Importance Analyzer

## Quick Start

**To analyze a model:**
1. Run cells in order starting from Cell 3 (imports)
2. Cell 5 shows all available models
3. Cell 6: Change `PATTERN` to select which model to analyze
4. Continue running cells sequentially

## Switch Models

**Method 1: Pattern Matching (Easiest)** ⭐
```python
# In Cell 6, just change the PATTERN:
PATTERN = 'RB_rush_yds_yards'  # Switch to RB rushing yards
```

**Method 2: By Index**
```python
# Uncomment and use index from Cell 5:
MODEL_INDEX = 2  # Analyze model [2] from the list
MODEL_PATH = str(models[MODEL_INDEX])
```

**Method 3: Most Recent**
```python
# Uncomment to auto-select newest model:
MODEL_PATH = str(models[0])
```

---


# Player Prop Feature Importance Analysis

Analyze feature importance from trained player prop models.

**Features:**
- Works with both RandomForest and XGBoost
- Feature importance rankings
- Correlation analysis (find redundant features)
- Feature selection recommendations
- Multiple visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Import analyzer (with reload to get latest version)
import importlib
import src.evaluation.player_prop_analysis.player_prop_feature_Importance.player_prop_feature_importance as feature_analyzer_module
importlib.reload(feature_analyzer_module)
from src.evaluation.player_prop_analysis.player_prop_feature_Importance.player_prop_feature_importance import FeatureAnalyzer

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300


## 🔧 Configuration

**Quick Start:**
1. Run the cell below to see all available models
2. In the next cell, change the `PATTERN` to the model you want to analyze
3. Run that cell, then continue with the rest of the notebook


In [ ]:
# ==================== STEP 1: View Available Models ====================
from pathlib import Path

project_root = Path(r'c:\NFL Predictive Model\Model\NFL_Model')
model_dir = project_root / 'artifacts' / 'player_props'

if model_dir.exists():
    models = sorted(model_dir.glob('*.pkl'), key=lambda x: x.stat().st_mtime, reverse=True)
    if models:
        print("📦 Available Models (most recent first):")
        print("=" * 90)
        for i, model in enumerate(models):
            size_mb = model.stat().st_size / (1024 * 1024)
            parts = model.stem.split('_')
            if len(parts) >= 5:
                position = parts[0]
                prop = parts[1] + '_' + parts[2]
                target = parts[3]
                print(f"  [{i}] {position:15s} {prop:20s} {target:20s} ({size_mb:.2f} MB)")
            else:
                print(f"  [{i}] {model.name} ({size_mb:.2f} MB)")
        print("=" * 90)
    else:
        print("⚠️  No trained models found in artifacts/player_props/")
        print("   Run: python src/scripts/player_prop/train_player_prop.py")
else:
    print(f"⚠️  Directory '{model_dir}' does not exist")
    print("   Run: python src/scripts/player_prop/train_player_prop.py")


In [ ]:
# ==================== STEP 2: Select Model to Analyze ====================

# 🎯 CHOOSE ONE METHOD - Uncomment the method you want to use:

# ─────────────────────────────────────────────────────────────────────────
# METHOD 1: Pattern Matching (Recommended - automatic model selection)
# ─────────────────────────────────────────────────────────────────────────
PATTERN = 'QB_pass_yds_yards'  # ← Change this to switch models!

# Available patterns:
#   'QB_pass_yds_yards'           - QB passing yards
#   'QB_pass_yds_td_probability'  - QB passing TD
#   'RB_rush_yds_yards'           - RB rushing yards  
#   'RB_any_td_any_td'            - RB any TD
#   'PASS_CATCHER_rec_yds_yards'  - WR/TE receiving yards
#   'PASS_CATCHER_rec_yds_td'     - WR/TE receiving TD
#   'PASS_CATCHER_receptions'     - WR/TE receptions

matching_models = [m for m in models if PATTERN in m.name]
if matching_models:
    MODEL_PATH = str(matching_models[0])
else:
    raise FileNotFoundError(f"No models found matching: '{PATTERN}'")

# ─────────────────────────────────────────────────────────────────────────
# METHOD 2: Index Selection (use index numbers from cell above)
# ─────────────────────────────────────────────────────────────────────────
# MODEL_INDEX = 0  # ← Change to [0], [1], [2], etc. from list above
# MODEL_PATH = str(models[MODEL_INDEX])

# ─────────────────────────────────────────────────────────────────────────
# METHOD 3: Auto-select most recent model
# ─────────────────────────────────────────────────────────────────────────
# MODEL_PATH = str(models[0])

# ═════════════════════════════════════════════════════════════════════════

print(f"✅ Selected: {Path(MODEL_PATH).name}\n")

# Analysis settings
OUTPUT_DIR = str(project_root / 'artifacts' / 'analysis')
CORRELATION_THRESHOLD = 0.90  # Features correlated above this are considered redundant


---

## 📊 Analysis

### 1. Load Model


In [ ]:
# Initialize analyzer
analyzer = FeatureAnalyzer(MODEL_PATH)

print(f"Model: {analyzer.model_name}")
print(f"Type: {getattr(analyzer, 'model_type', type(analyzer.model).__name__)}")
print(f"Total features: {len(analyzer.features)}")
print(f"Position: {analyzer.artifact.position}")
print(f"Prop: {analyzer.artifact.prop_type}")
print(f"Target: {analyzer.artifact.target_type}")


## 2. Feature Importance Rankings

In [ ]:
# Get importance data
importance_df = analyzer.get_importance()

# Add missing percentage columns if they don't exist
if 'importance_pct' not in importance_df.columns:
    importance_df['importance_pct'] = (importance_df['importance'] / importance_df['importance'].sum() * 100).round(2)
    
if 'cumulative_pct' not in importance_df.columns:
    importance_df['cumulative_pct'] = importance_df['importance_pct'].cumsum().round(2)

# Add categories
importance_df = analyzer.add_categories(importance_df)

# Display top 20
print("\n📊 TOP 20 FEATURES:\n")
display_cols = ['rank', 'feature', 'importance', 'importance_pct', 'cumulative_pct', 'category']
print(importance_df[display_cols].head(20).to_string(index=False))


In [ ]:
# Category summary
print("\n📊 IMPORTANCE BY CATEGORY:\n")
category_summary = analyzer.summarize_by_category()
print(category_summary)

## 3. Visualizations

In [ ]:
# Top 25 features bar chart
analyzer.plot_top_features(n=25)

In [ ]:
# Category breakdown
analyzer.plot_categories()

In [ ]:
# Cumulative importance curve
analyzer.plot_cumulative_importance()

# Find how many features for 90%
top_90_features = analyzer.select_top_features(cumulative_pct=90)
print(f"\n💡 {len(top_90_features)} features explain 90% of importance")
print(f"💡 Reduction potential: {len(analyzer.features)} → {len(top_90_features)} features")

## 4. Get Training Data for Correlation Analysis

**You need X_train to run correlation analysis.**

Two options:
1. Load saved training data (if you saved it during training)
2. Recreate from model workflow (shown below)

In [ ]:
# Option 2: Recreate training data from model workflow
from src.models.player_prop_model import NFLPlayerPropModel

# Recreate the model
model_recreate = NFLPlayerPropModel(
    position=analyzer.artifact.position,
    prop_type=analyzer.artifact.prop_type,
    target_type=analyzer.artifact.target_type
)

# Run the feature pipeline
print("Loading data...")
model_recreate.load_player_games(2018)

print("Building features...")
model_recreate.build_feature_matrices()

print("Building dataset...")
model_recreate.build_dataset()

# Get training split
X_train, X_test, y_train, y_test, features, is_class = model_recreate._select_X_y()

print(f"\n✅ Loaded training data: {X_train.shape}")
print(f"   Features: {X_train.shape[1]}")
print(f"   Samples: {X_train.shape[0]}")

## 5. Correlation Analysis

In [ ]:
# Find redundant features (correlation > 0.90)
print("\n🔍 FINDING REDUNDANT FEATURES (correlation > 0.90)...\n")
redundant_pairs = analyzer.find_redundant_features(X_train, threshold=CORRELATION_THRESHOLD)

print(f"Found {len(redundant_pairs)} highly correlated pairs")

# Show top 15
if redundant_pairs:
    print("\n TOP 15 MOST REDUNDANT PAIRS:")
    print("-" * 80)
    for feat1, feat2, corr in redundant_pairs[:15]:
        print(f"  {corr:6.3f}  {feat1:40s} <-> {feat2}")

In [ ]:
# Get removal recommendations
recommendations = analyzer.recommend_features_to_remove(X_train, threshold=CORRELATION_THRESHOLD)

print(f"\n💡 FEATURE SELECTION RECOMMENDATION:")
print(f"   Current: {len(analyzer.features)} features")
print(f"   Keep: {recommendations['kept_count']} features")
print(f"   Remove: {recommendations['removed_count']} redundant features")
print(f"   Reduction: {100 * recommendations['removed_count'] / len(analyzer.features):.1f}%")

if recommendations['remove']:
    print(f"\n🗑️  RECOMMENDED TO REMOVE ({len(recommendations['remove'])} features):")
    for feat in recommendations['remove'][:25]:
        print(f"   - {feat}")
    if len(recommendations['remove']) > 25:
        print(f"   ... and {len(recommendations['remove']) - 25} more")

In [ ]:
# Correlation heatmap for top 30 features
print("\n📊 Correlation heatmap for top 30 features...")
analyzer.plot_correlation_heatmap(X_train, top_n=30)

## 6. Category Deep Dive

In [ ]:
# Detailed category analysis
category_summary = analyzer.summarize_by_category()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total importance
category_summary['total_importance'].plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Total Importance by Category', fontweight='bold')
axes[0,0].set_xlabel('Total Importance')

# Average importance per feature
category_summary['avg_importance'].plot(kind='barh', ax=axes[0,1], color='coral')
axes[0,1].set_title('Average Importance by Category', fontweight='bold')
axes[0,1].set_xlabel('Average Importance')

# Feature count
category_summary['count'].plot(kind='barh', ax=axes[1,0], color='mediumseagreen')
axes[1,0].set_title('Number of Features by Category', fontweight='bold')
axes[1,0].set_xlabel('Count')

# Pie chart
axes[1,1].pie(
    category_summary['total_pct'], 
    labels=category_summary.index,
    autopct='%1.1f%%',
    startangle=90
)
axes[1,1].set_title('Importance Distribution', fontweight='bold')

plt.suptitle(f'Category Analysis - {analyzer.model_name}', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 7. Rolling Window Comparison

Compare roll3 vs roll4 vs roll5

In [ ]:
# Analyze rolling features
rolling_features = importance_df[importance_df['category'] == 'rolling'].copy()

if len(rolling_features) > 0:
    # Extract window size
    rolling_features['window'] = rolling_features['feature'].str.extract(r'roll(\d+)')[0]
    rolling_features['base_stat'] = rolling_features['feature'].str.split('__').str[0]
    
    # Group by window
    window_importance = rolling_features.groupby('window')['importance'].agg(['sum', 'mean', 'count'])
    
    print("\n📊 ROLLING WINDOW COMPARISON:")
    print(window_importance)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    window_importance['sum'].plot(kind='bar', ax=ax1, color='steelblue')
    ax1.set_title('Total Importance by Window Size', fontweight='bold')
    ax1.set_xlabel('Window Size')
    ax1.set_ylabel('Total Importance')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
    
    window_importance['mean'].plot(kind='bar', ax=ax2, color='coral')
    ax2.set_title('Average Importance by Window Size', fontweight='bold')
    ax2.set_xlabel('Window Size')
    ax2.set_ylabel('Average Importance')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
    
    plt.suptitle('Rolling Window Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Check correlation between windows
    print("\n🔍 Checking correlation between different window sizes...")
    roll3_feats = rolling_features[rolling_features['window'] == '3']['feature'].tolist()
    roll4_feats = rolling_features[rolling_features['window'] == '4']['feature'].tolist()
    roll5_feats = rolling_features[rolling_features['window'] == '5']['feature'].tolist()
    
    all_roll_feats = roll3_feats + roll4_feats + roll5_feats
    available = [f for f in all_roll_feats if f in X_train.columns]
    
    if len(available) > 1:
        roll_corr = X_train[available].corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(roll_corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
                   square=True, linewidths=0.5, annot=False)
        plt.title('Rolling Window Feature Correlation', fontweight='bold', fontsize=12)
        plt.tight_layout()
        plt.show()
else:
    print("No rolling features found")

## 8. Historical vs Rolling Features

In [ ]:
# Compare feature types
hist_features = importance_df[importance_df['category'] == 'historical']
roll_features = importance_df[importance_df['category'] == 'rolling']

print(f"\n📊 HISTORICAL vs ROLLING:")
print(f"   Historical: {len(hist_features)} features, avg importance: {hist_features['importance'].mean():.2f}")
print(f"   Rolling: {len(roll_features)} features, avg importance: {roll_features['importance'].mean():.2f}")

if len(hist_features) > 0:
    print(f"\n   Top historical: {hist_features.iloc[0]['feature']} (rank #{hist_features.iloc[0]['rank']:.0f})")
if len(roll_features) > 0:
    print(f"   Top rolling: {roll_features.iloc[0]['feature']} (rank #{roll_features.iloc[0]['rank']:.0f})")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

comparison_data = pd.DataFrame({
    'Type': ['Historical', 'Rolling'],
    'Total Importance': [
        hist_features['importance'].sum() if len(hist_features) > 0 else 0,
        roll_features['importance'].sum() if len(roll_features) > 0 else 0
    ],
    'Count': [len(hist_features), len(roll_features)]
})

x = np.arange(len(comparison_data))
width = 0.35

ax.bar(x - width/2, comparison_data['Total Importance'], width, label='Total Importance', color='steelblue')
ax2 = ax.twinx()
ax2.bar(x + width/2, comparison_data['Count'], width, label='Feature Count', color='coral')

ax.set_xlabel('Feature Type', fontweight='bold')
ax.set_ylabel('Total Importance', color='steelblue', fontweight='bold')
ax2.set_ylabel('Feature Count', color='coral', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_data['Type'])
ax.set_title('Historical vs Rolling Features', fontsize=12, fontweight='bold')

ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 9. Feature Selection Recommendations

In [ ]:
print("\n" + "="*80)
print("FEATURE SELECTION RECOMMENDATIONS")
print("="*80)

# Current state
print(f"\n📊 CURRENT STATE:")
print(f"   Total features: {len(analyzer.features)}")
print(f"   Model type: {analyzer.model_type}")

# Option 1: Top N
for n in [30, 50, 75]:
    if n <= len(analyzer.features):
        top_n = analyzer.select_top_features(n=n)
        pct = importance_df.head(n)['cumulative_pct'].iloc[-1]
        reduction = 100 * (1 - n / len(analyzer.features))
        print(f"\n💡 OPTION: Keep top {n} features")
        print(f"   Importance retained: {pct:.1f}%")
        print(f"   Reduction: {len(analyzer.features)} → {n} ({reduction:.0f}% fewer)")

# Option 2: Cumulative threshold
for pct_threshold in [85, 90, 95]:
    top_pct = analyzer.select_top_features(cumulative_pct=pct_threshold)
    reduction = 100 * (1 - len(top_pct) / len(analyzer.features))
    print(f"\n💡 OPTION: {pct_threshold}% importance threshold")
    print(f"   Features needed: {len(top_pct)}")
    print(f"   Reduction: {len(analyzer.features)} → {len(top_pct)} ({reduction:.0f}% fewer)")

# Option 3: Remove redundant
if recommendations['removed_count'] > 0:
    reduction = 100 * (recommendations['removed_count'] / len(analyzer.features))
    print(f"\n💡 OPTION: Remove redundant (r > {CORRELATION_THRESHOLD})")
    print(f"   Features to remove: {recommendations['removed_count']}")
    print(f"   Reduction: {len(analyzer.features)} → {recommendations['kept_count']} ({reduction:.0f}% fewer)")

# Recommendation
print(f"\n✅ RECOMMENDED APPROACH:")
print(f"   1. Start by removing redundant features (r > 0.90)")
print(f"   2. Then select top 50-75 by importance")
print(f"   3. Retrain model and compare metrics")
print(f"   4. If similar performance, use reduced set")
print("\n" + "="*80 + "\n")

## 10. Export Feature Lists

In [ ]:
# Create output directory
export_dir = Path(OUTPUT_DIR) / 'feature_selection'
export_dir.mkdir(parents=True, exist_ok=True)

# Export different feature sets
feature_sets = {}

# Top 50
if len(analyzer.features) >= 50:
    feature_sets['top50'] = analyzer.select_top_features(n=50)

# Top 75
if len(analyzer.features) >= 75:
    feature_sets['top75'] = analyzer.select_top_features(n=75)

# 90% threshold
top_90 = analyzer.select_top_features(cumulative_pct=90)
if len(top_90) > 0:
    feature_sets['top90pct'] = top_90

# After removing redundant
if recommendations['kept_count'] > 0:
    feature_sets['no_redundant'] = recommendations['keep']

# Save
for name, features in feature_sets.items():
    filepath = export_dir / f'{analyzer.model_name}_{name}_features.txt'
    pd.DataFrame({'feature': features}).to_csv(filepath, index=False, header=False)
    print(f"✅ Saved: {filepath.name} ({len(features)} features)")

print(f"\n📁 Feature lists saved to: {export_dir}/")

## 11. Generate Full Report

In [ ]:
# Generate comprehensive report with all visualizations
print("\n" + "="*80)
print("GENERATING FULL REPORT")
print("="*80 + "\n")

analyzer.save_report(
    X_train=X_train,
    output_dir=OUTPUT_DIR,
    correlation_threshold=CORRELATION_THRESHOLD
)

## 12. Summary

In [ ]:
# Print final summary
total = len(analyzer.features)
top20_pct = importance_df.head(20)['cumulative_pct'].iloc[-1]
redundant_count = len(redundant_pairs)

top_cats = category_summary.head(3)

print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)
print(f"\n📊 Feature Overview:")
print(f"   Total features: {total}")
print(f"   Top 20 explain: {top20_pct:.1f}% of importance")
print(f"   Redundant pairs found: {redundant_count}")

print(f"\n📊 Most Important Categories:")
for i, (cat, row) in enumerate(top_cats.iterrows(), 1):
    print(f"   {i}. {cat.title()}: {row['total_pct']:.1f}% importance, {int(row['count'])} features")

print(f"\n💡 Key Recommendations:")
if recommendations['removed_count'] > 0:
    print(f"   - Remove {recommendations['removed_count']} redundant features")
    print(f"   - Keep top 50-75 features (85-90% importance)")
    print(f"   - Potential reduction: {total} → ~60 features")
else:
    print(f"   - Keep top 50-75 features")
    print(f"   - Potential reduction: {total} → 50-75 features")

print("\n" + "="*80 + "\n")

print("✅ Analysis complete!")
print(f"📁 Reports saved to: {OUTPUT_DIR}/")
print(f"📁 Feature lists saved to: {export_dir}/")

## Next Steps

### Use Selected Features for Retraining:

```python
# Load selected features
with open('artifacts/analysis/feature_selection/model_top50_features.txt') as f:
    selected_features = [line.strip() for line in f]

# Train new model with reduced features
from src.models.player_prop_model import NFLPlayerPropModel

model_v2 = NFLPlayerPropModel(
    position='QB',
    prop_type='pass_yds',
    target_type='yards',
    stats=selected_features  # Use selected features only
)

model_v2.load_player_games(2018)
model_v2.build_feature_matrices()
model_v2.build_dataset()
metrics_v2 = model_v2.fit_xgb()

# Compare to original
print(f"Original: {metrics_original}")
print(f"Reduced:  {metrics_v2}")
```

### If Metrics Are Similar:
- Use reduced model (faster training, less overfitting)
- Simpler to maintain and explain
- Better generalization

### If Metrics Drop Significantly:
- Try different feature set (e.g., top 75 instead of top 50)
- Try removing only redundant features
- Consider keeping all features